# Classification of EEG data using raw data 

In this notebook, instead of simply using band features -- using Welch's method -- to create our features, I wanted to use the raw datapoints. The idea is to break the data into 'windows' of timepoints where there are more or less datapoints in general. Using this method would give a different perspective as to how this data can perform -- maybe it will do better with the raw data instead of looking at different frequency bands. How will the window size affect the accuracy of the models?

The following code is used from the pipeline to do some analysis on the data:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import mne

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:

# The code to find the project root for the file path
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")


# These are the specific file paths for the project
PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "nm000114"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_features"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

# Get one specific eeg file loaded
def get_eeg_file(subject_id: str, condition: str):
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

PROJECT_ROOT: c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-
RAW_DATA_DIR: c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114
PROCESSED_DIR: c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\processed_features
RESULTS_DIR: c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\results


In [3]:
# Get a specific file
file_path = get_eeg_file("sub-HS1", "P300")

print(type(file_path))
print(file_path.exists())

# Load the data for the first file
raw = mne.io.read_raw_edf(file_path, preload=True)

# Gets turned into raw numpy array
data = raw.get_data()
print("Shape: ", data.shape) # 22 channels with 154880 values in each (A 2D array)

# Get the timepoints for channel 0 
channel_0 = data[0, :] 

# Gives us the values in channel 0
print(channel_0[10:], len(channel_0))


# 22 channels and 154880 timestamps

<class 'pathlib._local.WindowsPath'>
True
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-P300_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 154879  =      0.000 ...   604.996 secs...
Shape:  (22, 154880)
[7.65058366e-06 2.45018692e-06 3.15024033e-06 ... 1.67512779e-05
 1.08508278e-05 5.45041581e-06] 154880


In [4]:
# These are all the channels they should have in common
COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]

At this point, I want to know whether all the files have the same shape -- what their format is to see how I could generate them

In [6]:
edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

In [7]:
# Parser cell to read all EDF files and extract data and labels for modeling

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition from the filename
    Returns:
        - eyesClosed
        - eyesOpen
        - P300
    """

    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")

In [8]:
# Get all the eeg files 
def get_all_eegs(files):
    raws = []
    for file in files:
        raw = mne.io.read_raw_edf(file, preload=True)
        raws.append(raw)

    return raws

all_eegs = get_all_eegs(edf_files)
    

Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesClosed_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 76799  =      0.000 ...   299.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesOpen_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 89855  =      0.000 ...   350.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-P300_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 154879  =      0.000 ...   604.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-projec

In [9]:
print(f"There are {len(all_eegs)} eeg files")

# Get the channels, timepoints, and sample frequency (how many samples per second)
n_channels = []
n_timepoints = []
sfreqs = []


for raw in all_eegs:
    sfreqs.append(raw.info['sfreq'])
    data = raw.get_data()
    n_channels.append(data.shape[0])
    n_timepoints.append(data.shape[1])


# Print a bunch of information to try to understand the structure of the data
print(f"The mean sample frequency is {np.mean(sfreqs)}")
print(f"The mean amount of channels is {np.mean(n_channels)}")
print(f"The mean amount of timepoints is {np.mean(n_timepoints)}")
print(f"The std amount of timepoints is {np.std(n_timepoints)}")

print(f"The unique sample frequencies is {set(sfreqs)}")
print(f"The unique number of channel counts are {set(n_channels)}")
print(f"The unique number of timepoints are {set(n_timepoints)}")

print(f"Max number of timepoint: {max(n_timepoints)}")
print(f"Min number of timepoint: {min(n_timepoints)}")


There are 181 eeg files
The mean sample frequency is 256.0
The mean amount of channels is 21.23756906077348
The mean amount of timepoints is 104664.39779005524
The std amount of timepoints is 39737.60344141525
The unique sample frequencies is {256.0}
The unique number of channel counts are {20, 22}
The unique number of timepoints are {76800, 89856, 96256, 154880, 77312, 77568, 156160, 49664, 155392, 74752, 78080, 155904, 164096, 61696, 168448, 78592, 164608, 156416, 160512, 155648, 164864, 156672, 91136, 160768, 46080, 79104, 161024, 157184, 75520, 79616, 161536, 79872, 161792, 75776, 76032, 162048, 153856, 166144, 162304, 76288, 81920, 76544, 162560, 154624, 158720, 162816, 77056, 163072, 48384, 175616, 163328, 163584}
Max number of timepoint: 175616
Min number of timepoint: 46080


There is a wide amount of timepoints, meaning there is no consistency for classification. By using windows, I can potentially do 'data augmentation' and see whether artificially having more data points could increase the accuracy of the model

In [15]:
# metadata list to hold all parsed information 
rows = []

# Scan all files in the raw data directory to ensure we can access them
edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

for filepath in edf_files:
    subject_id = filepath.parts[-3]  # e.g. subHS1
    condition = parse_condition_from_filename(filepath)
    label = infer_label_from_subject(subject_id)

    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": label,
        "condition": condition,
        "filepath": str(filepath),
    })

# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)

This code is to make the windows, which can increase depending on how much I multiply sfreq by (1 second)

In [ ]:
def make_windows(raw, label, group_id):
    # This data will be in the (channels, time) format
    data_raw = raw.copy()

    # This makes sure the format is consistent at 20 channels
    data_raw.pick(COMMON_CHANNELS) 

    # get data
    data = data_raw.get_data()      # shape: (n_channels, n_time points)
    sfreq = int(data_raw.info['sfreq'])  # sampling frequency (e.g. 256 Hz)

    n_channels, n_timepoints = data.shape
    window_size = sfreq * 100 # 1 second

    X_chunks = []
    y_chunks = []
    groups = []

    # Get the number of windows without including the last one
    n_windows = n_timepoints // window_size

    # Seperate the windows of data depending on window_size
    for i in range(n_windows):
        start = i * window_size
        end = start + window_size

        window = data[:, start:end] # (channels, sfreq)

        # Flatten the features
        features = window.flatten() # (timepoints, flattened data)
        
        X_chunks.append(features)
        y_chunks.append(label)
        groups.append(group_id)

    return X_chunks, y_chunks, groups

In [ ]:
all_X = []
all_y = []
all_groups = []

# Go through metadata_df and extract the info
for _, row in metadata_df.iterrows():
    filepath = row["filepath"]
    label = row["label"]
    patient_id = row["patient_id"]

    # Extract the raw data for the filepath
    raw = mne.io.read_raw_edf(filepath, preload=True)

    # Get the different windows of data from make_windows
    X_chunks, y_chunks, groups = make_windows(
        raw=raw,
        label=label,
        group_id=patient_id
    )

    # Add the data to the arrays
    all_X.extend(X_chunks)
    all_y.extend(y_chunks)
    all_groups.extend(groups)

# Turn into numpy arrays
X = np.array(all_X)
y = np.array(all_y)
groups = np.array(all_groups)

Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesClosed_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 76799  =      0.000 ...   299.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-eyesOpen_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 89855  =      0.000 ...   350.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-project-\data\raw\nm000114\sub-HS1\eeg\sub-HS1_task-P300_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 154879  =      0.000 ...   604.996 secs...
Extracting EDF parameters from c:\Users\AlexC\OneDrive\Desktop\berkeley_stuff\CHEM 277B\MSSE-277b-final-projec

This will give me a shape of the data to see how many datapoints are being analyzed

In [ ]:
# print the shapes to see the sizes
print(X.shape, y.shape, groups.shape)

# Print how many of each classification there are
print(np.bincount(y))

(694, 512000) (694,) (694,)
[333 361]


In [ ]:
# create the splitter, five folds total 
gfk = GroupKFold(n_splits=5)

# build a baseline pipeline with standard scaling and logistic regression 
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=100)), # Had to run PCA because otherwise the dataset was too large to fully run
    ("clf", LogisticRegression(max_iter=1000))
])


# evaluate the pipeline using cross-validation 
results_lr = cross_validate(
    pipeline, 
    X, 
    y,
    cv=gfk.split(X, y, groups=groups),
    scoring=["accuracy", "f1", "roc_auc"]
)

print("Accuracy:", results_lr["test_accuracy"])
print("Mean accuracy:", results_lr["test_accuracy"].mean())
print("Std accuracy:", results_lr["test_accuracy"].std())

print("F1 score:", results_lr["test_f1"])
print("Mean F1 score:", results_lr["test_f1"].mean())

print("ROC-AUC:", results_lr["test_roc_auc"])
print("Mean ROC-AUC:", results_lr["test_roc_auc"].mean())

Accuracy: [0.47857143 0.44285714 0.55714286 0.54285714 0.64179104]
Mean accuracy: 0.5326439232409381
Std accuracy: 0.06870677167229891
F1 score: [0.49655172 0.56179775 0.5974026  0.56756757 0.7037037 ]
Mean F1 score: 0.5854046691241577
ROC-AUC: [0.64583333 0.47306034 0.57084441 0.58758503 0.61195055]
Mean ROC-AUC: 0.5778547339443156


Clearly, this model is NOT working well

In [58]:
# Run cross-validation with SVM

pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=100)),
    ("clf", SVC(kernel="rbf",probability=True)
     )
])

results_svm = cross_validate(
    pipeline_svm,
    X,
    y,
    cv=gfk.split(X, y, groups),
    scoring=["accuracy", "f1", "roc_auc"]
)

print("SVM accuracy:", results_svm["test_accuracy"])
print("SVM mean accuracy:", results_svm["test_accuracy"].mean())
print("SVM std accuracy:", results_svm["test_accuracy"].std())

print("SVM F1 score:", results_svm["test_f1"])
print("SVM mean F1 score:", results_svm["test_f1"].mean())  

print("SVM ROC-AUC:", results_svm["test_roc_auc"])
print("SVM mean ROC-AUC:", results_svm["test_roc_auc"].mean())

SVM accuracy: [0.44285714 0.65       0.57142857 0.52857143 0.71641791]
SVM mean accuracy: 0.5818550106609808
SVM std accuracy: 0.09488853627326804
SVM F1 score: [0.53012048 0.78222222 0.6875     0.62921348 0.80412371]
SVM mean F1 score: 0.6866359797272412
SVM ROC-AUC: [0.6545928  0.45869253 0.65242282 0.56505102 0.68887363]
SVM mean ROC-AUC: 0.603926559193495


SVM seems to do about the same as logistic regression

Looking at the performance of this file, it seems that using the raw data is simply too noisy. When increasing the window size, the data does slightly better, but the performance barely ever breaks 60%. It seems like, at least for this dataset, using the raw data doesn't seem to work very well.